# 06 — Resultados Finais e Conclusões

Consolidação dos principais achados do projeto de análise de atrasos de voos nos EUA.

## 1. Resumo da Análise Exploratória (EDA)

**Principais achados:**
- O dataset contém ~5.8 milhões de voos domésticos nos EUA (2015).
- A distribuição de atrasos é fortemente assimétrica à direita: a maioria dos voos parte no horário ou com pequenos atrasos, mas alguns apresentam atrasos extremos (horas).
- Meses de verão (junho-agosto) e dezembro concentram os maiores atrasos médios, correlacionados com aumento de demanda e condições meteorológicas.
- Sextas-feiras e domingos são os dias da semana com mais atrasos.
- Existe forte correlação entre `DEPARTURE_DELAY` e `ARRIVAL_DELAY`, indicando que atrasos na partida propagam para a chegada.
- Os grandes hubs (ATL, ORD, LAX, DFW) concentram o maior volume absoluto de atrasos.

## 2. Modelagem Supervisionada — Classificação de Atrasos

**Abordagem:** Classificação binária — prever se um voo vai atrasar 15+ minutos (padrão FAA).

**Modelos comparados:** Logistic Regression, Random Forest, Gradient Boosting — cada um em versão padrão e com balanceamento de classes (`class_weight='balanced'` / `sample_weight`).

**Resultados:**
- Os modelos baseados em árvores superam a Logistic Regression, indicando relações não-lineares no dataset.
- **Os modelos balanceados melhoram drasticamente o recall** (capacidade de detectar voos atrasados), que era o ponto fraco dos modelos padrão (3-10% de recall).
- A curva **Precision-Recall** (PR-AUC) se mostrou mais informativa que o ROC-AUC para este problema desbalanceado (82% vs 18%), revelando diferenças entre modelos que o ROC-AUC mascara.

**Análise de threshold:**
- O threshold padrão de 0.5 é inadequado para dados desbalanceados. Ao ajustar o threshold, é possível obter um melhor equilíbrio entre precision e recall conforme a necessidade operacional.

**Análise de custo de erro:**
- Em aviação, o custo de **não prever um atraso** (falso negativo — passageiro perde conexão) é maior que o custo de um **alerta falso** (falso positivo — passageiro se prepara desnecessariamente). Isso justifica priorizar recall sobre precision na escolha do modelo e threshold.

## 3. Modelagem Não Supervisionada — Clustering de Aeroportos

**Abordagem:** K-Means clustering no nível de aeroporto (~320 aeroportos), usando métricas agregadas: atraso médio, taxa de cancelamento, volume de voos, distância média.

**Resultados:**
- 4 clusters distintos foram identificados, representando perfis operacionais diferentes:
  - Grandes hubs com tráfego intenso
  - Aeroportos regionais com operação limitada
  - Aeroportos com altas taxas de atraso
  - Aeroportos eficientes com baixo atraso
- A projeção PCA 2D mostrou separação razoável entre os clusters.
- O silhouette score confirma que os clusters têm coesão moderada.

## 4. Limitações

1. **Dados de apenas 1 ano (2015):** Padrões sazonais podem variar entre anos. Múltiplos anos permitiriam validação mais robusta.
2. **Ausência de dados meteorológicos:** Clima é a principal causa de atrasos severos e não está presente no dataset. Integrar dados de APIs meteorológicas melhoraria significativamente a previsão.
3. **Desbalanceamento de classes:** Embora o uso de `class_weight='balanced'` tenha melhorado substancialmente o recall, técnicas adicionais como SMOTE ou undersampling poderiam ser exploradas.
4. **Label encoding para modelos lineares:** A Logistic Regression sofre com label encoding de variáveis categóricas de alta cardinalidade (320+ aeroportos). One-hot encoding ou target encoding seriam mais adequados.
5. **Granularidade temporal limitada:** Horários em formato HHMM inteiro perdem informação. Dados com timestamps precisos permitiriam features mais ricas (ex.: tempo entre voos consecutivos no mesmo gate).
6. **Validação temporal não aplicada:** O split treino/teste é aleatório. Uma validação temporal (treinar em meses anteriores, testar em posteriores) seria mais realista para um cenário de produção.

## 5. Melhorias e Próximos Passos

1. **Incorporar dados meteorológicos:** Usar APIs como NOAA/Weather para adicionar temperatura, precipitação e visibilidade como features.
2. **Modelos mais sofisticados:** XGBoost, LightGBM ou redes neurais para capturar padrões complexos.
3. **Feature engineering avançada:** Atraso acumulado no aeroporto nas últimas horas, histórico de atrasos da aeronave (TAIL_NUMBER), congestionamento por faixa horária.
4. **Validação temporal:** Usar split temporal (treinar em meses anteriores, testar em posteriores) em vez de split aleatório.
5. **Dashboard interativo:** Criar visualização com Plotly Dash ou Streamlit para exploração dinâmica dos dados.
6. **Detecção de anomalias:** Identificar voos com atrasos atípicos usando Isolation Forest ou DBSCAN.
7. **Pipeline de produção:** Empacotar o modelo como API REST para previsão em tempo real.